# Week 36, Day 3: The Kalman filter: decoding movement over time

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w036/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Fitting A, W, H and Q

Fit A by least squares so that X[t+1] is approximately A @ X[t] on the training data, then W as the covariance of the residuals. H, c and Q are fitted for you.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(2006)
dt, T, N = 0.05, 12000, 40
X = np.zeros((T, 4))                           # x, y, vx, vy
for t in range(1, T):
    p, v = X[t - 1, :2], X[t - 1, 2:]
    v = v + dt * (-2.0 * v - 1.5 * p) + np.sqrt(dt) * 12 * r.normal(size=2)
    X[t, :2] = p + dt * v; X[t, 2:] = v
pdirs = r.uniform(0, 2 * np.pi, N); gain = r.uniform(0.5, 1.5, N); base = r.uniform(5, 20, N)
pw = r.normal(0, 0.3, (N, 2))
rate = np.maximum(base + gain * (X[:, 2:3] * np.cos(pdirs) + X[:, 3:4] * np.sin(pdirs)) + X[:, :2] @ pw.T, 0)
Z = r.poisson(rate * dt).astype(float)
tr = np.arange(T) < 8000; te = ~tr
r2 = lambda a, b: 1 - ((a - b) ** 2).sum(0) / ((a - a.mean(0)) ** 2).sum(0)

Xtr = X[tr]
A = ...
W = ...
Hc = np.linalg.lstsq(np.c_[Xtr, np.ones(len(Xtr))], Z[tr], rcond=None)[0].T
H, c = Hc[:, :4], Hc[:, 4]
Q = np.cov((Z[tr] - np.c_[Xtr, np.ones(len(Xtr))] @ Hc.T).T)
print("A =\n", np.round(A, 3))

In [ ]:
import numpy as np
def test_shapes():
    "A is 4 by 4, H is 40 by 4 and Q is 40 by 40"
    assert A.shape == (4, 4) and H.shape == (40, 4) and Q.shape == (40, 40)

def test_dynamics():
    "A predicts the next state's velocity with R squared above 0.8 on the test set"
    pred = X[te][:-1] @ A.T
    assert r2(X[te][1:], pred)[2:].min() > 0.8

test_shapes()
test_dynamics()
print("All checks passed")

### Exercise: Decoding the test set

Complete the update step of kalman(Zs): compute S, the gain K, the new estimate x and covariance P, following the three equations above.

In [ ]:
def kalman(Zs):
    x, P, out = np.zeros(4), np.eye(4) * 10, []
    for z in Zs:
        x = A @ x
        P = A @ P @ A.T + W
        ...
        out.append(x.copy())
    return np.array(out)

Xk = kalman(Z[te])
R2 = {"Kalman": r2(X[te], Xk)}
print("Kalman R squared (x, y, vx, vy):", np.round(R2["Kalman"], 3))
plt.plot(X[te][:400, 0], "k", label="true x"); plt.plot(Xk[:400, 0], label="Kalman"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_velocity():
    "the Kalman filter decodes velocity with R squared above 0.75"
    assert R2["Kalman"][2:].min() > 0.75

def test_position():
    "it decodes position with R squared above 0.85"
    assert R2["Kalman"][:2].min() > 0.85

test_velocity()
test_position()
print("All checks passed")

### Exercise: Three decoders

Fit the Wiener filter: build Zl, the counts at lags 0 to 9 side by side (rows before lag 10 set to zero), add a constant, fit ridge weights with penalty 1 on the training bins, and store the test R² in R2['Wiener'].

In [ ]:
Z1 = np.c_[Z, np.ones(T)]
B = np.linalg.lstsq(Z1[tr], X[tr], rcond=None)[0]
R2["OLE"] = r2(X[te], Z1[te] @ B)
Lg = 10
Zl = np.concatenate([np.roll(Z, k, 0) for k in range(Lg)], 1); Zl[:Lg] = 0
Zl1 = np.c_[Zl, np.ones(T)]
Bw = ...
R2["Wiener"] = r2(X[te], Zl1[te] @ Bw)
for k in ("OLE", "Wiener", "Kalman"):
    print(f"{k:7s} position {R2[k][:2].mean():.2f}  velocity {R2[k][2:].mean():.2f}")

In [ ]:
import numpy as np
def test_ole_position():
    "the current-bin linear decoder decodes position poorly (R squared below 0.4)"
    assert R2["OLE"][:2].max() < 0.4

def test_order():
    "the Kalman filter beats the Wiener filter on position by more than 0.1"
    assert R2["Kalman"][:2].mean() > R2["Wiener"][:2].mean() + 0.1

test_ole_position()
test_order()
print("All checks passed")